# LAB03 — Word Representations and Embeddings
Core: co-occurrence → cosine → Skip-gram Word2Vec (numpy) → eval → semantic search.
Gensim không build được trên Python 3.14 (cần MSVC) nên dùng `w2v_numpy.py`: Skip-gram + negative sampling chuẩn.

In [ ]:
from cooccurrence import build_vocabulary, build_cooccurrence_matrix, cosine_similarity, most_similar, describe_matrix
import numpy as np

corpus = [
    'the cat eats fish',
    'the dog eats fish',
    'the cat likes milk',
    'the dog likes meat',
]
toks = [s.lower().split() for s in corpus]
for w in [1,2,5]:
    vocab, w2i = build_vocabulary(toks)
    X = build_cooccurrence_matrix(toks, vocab, w2i, window=w)
    print(f'window={w}', describe_matrix(X))
    print(' cos(cat,dog)=', round(cosine_similarity(X[w2i["cat"]], X[w2i["dog"]]),4))
    print(' most_similar(cat)=', most_similar('cat', X, vocab, w2i, 3))


Kết quả chạy thật: window=1 → nnz=18, cos(cat,dog)=1.0; window=2/5 → nnz=30/36, cos=0.875 (câu dài 4 từ nên window>=2 bão hòa).

In [ ]:
from w2v_numpy import build_vocab, get_training_pairs, SkipGramNEG, cosine, most_similar as ms, analogy, build_demo_corpus
import time

tok = build_demo_corpus(repeat=12)
vocab, w2i, cnt, freq = build_vocab(tok, min_count=2)
print('V=', len(vocab))
pairs5 = get_training_pairs(tok, w2i, window=5)
print('pairs win5=', len(pairs5))

t0=time.time()
model = SkipGramNEG(len(vocab), dim=50, seed=42).train(pairs5, freq, epochs=8, lr=0.05, negative=5, seed=42)
print(f'train {time.time()-t0:.1f}s')
V = model.vectors
print('most_similar(doctor)=', ms('doctor', vocab, w2i, V, 5))
def sim(a,b): return cosine(V[w2i[a]], V[w2i[b]])
for p in [('doctor','physician'),('car','automobile'),('king','queen'),('cat','dog'),('computer','banana')]:
    print(p, round(sim(*p),4))
print('analogy king-man+woman=', analogy(['king','woman'],['man'], vocab, w2i, V, 5)[0])


Main model (dim=50, win=5): doctor-physician 0.980 (rank1), cat-dog 0.965, car-auto 0.929, queen analogy rank1.

In [ ]:
# Experiment 3: window = 2,5,10 (dim=50, epochs=8)
for w in [2,5,10]:
    p = get_training_pairs(tok, w2i, window=w)
    m = SkipGramNEG(len(vocab), dim=50, seed=42).train(p, freq, epochs=8, lr=0.05, negative=5, seed=42)
    s = lambda a,b: cosine(m.vectors[w2i[a]], m.vectors[w2i[b]])
    print(f'win={w} doc-phys={s("doctor","physician"):.4f} cat-dog={s("cat","dog"):.4f} car-auto={s("car","automobile"):.4f} comp-ban={s("computer","banana"):.4f}')
    print('  analogy:', analogy(['king','woman'],['man'], vocab, w2i, m.vectors, 3)[0])


Nhận xét: win2 tách domain kém (computer-banana 0.81), win5/10 còn ~0.52; analogy win2 queen rank4, win5/10 rank1. Window lớn không luôn tốt (king-queen giảm ở win10).

In [ ]:
# Experiment 4: dim = 50,100,300 (win=5)
for d in [50,100,300]:
    m = SkipGramNEG(len(vocab), dim=d, seed=42).train(pairs5, freq, epochs=8, lr=0.05, negative=5, seed=42)
    s = lambda a,b: cosine(m.vectors[w2i[a]], m.vectors[w2i[b]])
    print(f'dim={d} sizeKB={m.vectors.nbytes/1024:.1f} doc-phys={s("doctor","physician"):.4f} cat-dog={s("cat","dog"):.4f} car-auto={s("car","automobile"):.4f}')


Dim 50→300: tăng marginal, size 30KB→178KB. Corpus nhỏ không lấp nổi chiều thừa → trade-off dimension↔capacity↔data↔compute.

In [ ]:
# Application: semantic search (query -> mean embedding -> cosine over docs)
docs = [
    'the doctor treated the patient with medical therapy',
    'clinical hospital treats disease with treatment',
    'the cat eats fish and likes milk',
    'i drive my car to the city',
    'football is a sport with a ball',
]
def doc_vec(t):
    ids=[w2i[w] for w in t.lower().split() if w in w2i]
    return V[ids].mean(axis=0) if ids else np.zeros(V.shape[1])
q = doc_vec('medical treatment')
rank = sorted([(d, float(q@doc_vec(d)/(np.linalg.norm(q)*np.linalg.norm(doc_vec(d))+1e-9))) for d in docs], key=lambda x:-x[1])
for d,s in rank: print(f'{s:.4f}  {d}')

# clustering demo
from sklearn.cluster import KMeans
words=['doctor','physician','hospital','nurse','patient','car','automobile','cat','dog','computer','banana','king','queen','football']
M=np.stack([V[w2i[w]] for w in words])
lab=KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(M)
for w,l in zip(words,lab): print(l,w)


Search `medical treatment` đưa 2 docs y tế lên top (therapy/clinical/patient gần query) dù không trùng từ — điều TF-IDF lexical không làm được. KMeans gom medical / vehicles-pets / royalty / còn lại, nhưng computer-banana đôi khi chung cluster → lỗi static embedding (xem error_analysis.md).